In [1]:
import pandas as pd
import numpy as np

try:
    final_df = pd.read_csv('../notebooks/clustering_tests/test_samples/TRAIN_D.csv')
    print(f"Загружен размеченный датасет: {len(final_df)} строк.")
    display(final_df.head())
except FileNotFoundError:
    print("Ошибка: файл 'final_labeled_dataset.csv' не найден. Сначала необходимо запустить блокнот 01-EDA-and-Clustering.")
    final_df = pd.DataFrame() 

try:
    full_df = pd.read_csv('../data/messages_processed.csv') 
    
    # Чтобы найти сообщения "Прочее", мы просто уберем из полного датасета
    
    #временный DataFrame для слияния
    temp_df = pd.merge(
        full_df, 
        final_df[['text']], # Нам нужна только одна колонка из final_df для идентификации
        on='text',          # Объединяем по колонке 'text'
        how='left',         # Left join
        indicator=True      # Это добавит колонку '_merge', которая покажет, где нашлась пара
    )
    
    # Отбираем только те строки, которые нашлись ТОЛЬКО в левом датасете (full_df),
    # то есть, не нашлись в final_df. Это и есть наши "Прочие".
    other_df = temp_df[temp_df['_merge'] == 'left_only'].copy()
    
    # Удаляем служебную колонку '_merge'
    other_df.drop(columns=['_merge'], inplace=True)
    
    print(f"\nИзвлечен датасет 'Прочее': {len(other_df)} строк.")
    display(other_df.head())

except FileNotFoundError:
    print("Ошибка: файл 'messages_processed.csv' не найден. Сначала необходимо запустить блокнот 01-EDA-and-Clustering.")
    other_df = pd.DataFrame()

Загружен размеченный датасет: 25698 строк.


,text,processed_text,pattern_name
0,Авария 10003\nИС: Deposits Back-Office\nОписан...,авария иса описание недоступность авторизация ...,API Авторизации (HTTP 500)
1,Авария 10005\nИС: Monitoring (Zabbix)\nОписани...,авария иса описание высокий загрузка узел назн...,Превышение порогов (Мониторинг)
2,Авария 10011\nИС: Payment Hub\nОписание: Потер...,авария иса описание потеря связь банкомат реги...,API Авторизации (HTTP 500)
3,Авария 10013\nИС: Deposits Back-Office\nОписан...,авария иса описание ошибка соединение бд назна...,Сбой соединения с Oracle (ORA-12514)
4,Авария 10017\nИС: 1C-Бухгалтерия\nОписание: Ош...,авария иса бухгалтерия описание ошибка запись ...,Ошибка записи на SMB-шару



Извлечен датасет 'Прочее': 20144 строк.


,id,text,text_len,processed_text
0,18789fab-71ae-4b8b-9a65-c653af3a1d94,Авария 10001\nИС: Keycloak IdP\nОписание: 1018...,122,авария иса описание назначать группа платежный...
1,7210f58d-88ba-491b-812b-9d4d150dcb65,Авария 10002\nИС: GitLab CI\nОписание: 3500\nН...,127,авария иса описание назначать группа операцион...
3,acdaea40-2637-4055-998d-026ad2f21148,Авария 10004\nИС: ElasticSearch\nОписание: 101...,121,авария иса описание назначать группа
5,d1fe1667-7889-4dc6-a8e6-747202f66ba2,Авария 10006\nИС: SWIFT Gateway\nОписание: Tim...,167,авария иса описание обращение соединение разры...
6,fc9bb86d-951a-40ea-9aba-d6a6d2593098,Авария 10007\nИС: ATM Monitoring System\nОписа...,183,авария иса описание сбой репликация минута наз...


In [31]:
INFRASTRUCTURE_KEYWORDS = {
    'zabbix': 5,'cpu': 5, 'heartbeat': 1, 'ram': 3, 'swap': 1, 'диск': 3, 'память': 3, 'загрузка': 1,
    'сеть': 3, 'firewall': 5, 'сервер': 3, 'узел': 3, 'mount point': 1, 
    'файловый': 1, 'репликация': 1, 'handshake': 1, 'snmp': 3, 'kubernetes': 5, 'pod': 3, 
    'postgresql': 3, 'oracle': 5, 'listener': 1, 'tablespace': 1, 'сетевой': 3, 'vpn': 5, 
    'gateway': 5, 'шлюз': 1, 'соединение': 1, 'инфраструктура': 5, 'устройство': 1, 'очередь': 3, 
    'переполнение': 3, 'заполнение': 3, 'отклик': 3, 'подключаться': 3, 'подключиться': 3, 
    'задержка': 1, 'файловый': 1, 'файл': 1, 'чтение': 1, 'источник': 3, 'цод': 5, 'гипервизор': 3, 
    'k8s': 5, 'sql': 3, 'mysql': 3, 'мсэ': 5, 'база': 3, 'бд': 3, 'температура': 1
}

APPLICATION_KEYWORDS = {
    'авторизация': 5, 'платеж': 5, 'транзакция': 5, 'отчет': 5, 'выписка': 5, 'система': 1, 'интерфейс': 3,
    'api': 3, 'http': 3, '500' :3, '400': 3, '401': 3, '402': 3, '404': 3,'порт': 3, 'форма': 3, 'кнопка': 3, 'логин': 3, 
    'пользователь': 3, 'сервис': 3, 'модуль': 3, 'функционал': 3, 'приложение': 3, 'ошибка': 1, '1c': 5,
    'бухгалтерия': 3, 'витрина': 3, 'dwh': 5, 'устранена' :1, 'устранить':1, 'инцидент': 1, 
    'группа': 1, 'флаг': 1, 'завершение': 3, 'внутренний': 3, 'отвечать': 1, 'просадка': 3,
    'падение': 1, 'утечка': 3, 'пул': 3, 'отлик': 3, 'назначать': 1, 'назначить': 1
}

ZOD_KEYWORDS = [
    'начать', 'выполнять', 'остановить', 'начинать'
]

EMAIL_KEYWORDS = [
    'отправить', 'отпралять'
]

def classify_message_type_priority(row):
    """
    Классификация по первому найденному ключевому слову с приоритетами.
    Принимает на вход целую строку DataFrame.
    """
    raw_text = str(row['text']).lower()
    processed_text = str(row['processed_text'])

    for keyword in ZOD_KEYWORDS:
        if processed_text.startswith(keyword):
            return "ZOD"

    if processed_text.startswith('отправлять'):
            return "EMAIL"

    #if any(keyword in raw_text for keyword in INFRASTRUCTURE_KEYWORDS):
        #return "Инфраструктурное"
    
    #if any(keyword in raw_text for keyword in APPLICATION_KEYWORDS):
        #return "Прикладное"
        
    #return "Неопределено"
    infra_score = sum(weight for keyword, weight in INFRASTRUCTURE_KEYWORDS.items() if keyword in raw_text)
    app_score = sum(weight for keyword, weight in APPLICATION_KEYWORDS.items() if keyword in raw_text)

    # Добавляем "порог" для принятия решения
    if infra_score == 0 and app_score == 0:
        return "Тестовое"
    
    if infra_score <= app_score:
        return "Инфраструктурное"
    else:
        return "Прикладное"

other_df['message_type'] = other_df.apply(classify_message_type_priority, axis=1)

print("Результаты классификации для сообщений 'Прочее':")
print(other_df['message_type'].value_counts())

def display_samples(df, message_type, n=5):
    print(f"\nПримеры сообщений типа '{message_type}':")
    samples = df[df['message_type'] == message_type]
    if not samples.empty:
        display(samples.sample(min(n, len(samples))))
    else:
        print(f"Сообщений типа '{message_type}' не найдено.")

display_samples(other_df, 'Инфраструктурное')
display_samples(other_df, 'Прикладное')
display_samples(other_df, 'ZOD')
display_samples(other_df, 'EMAIL')
display_samples(other_df, 'Тестовое')

other_df.to_csv('../data/other_classified_dataset.csv', index=False)
print("\nДатасет 'Прочее' с метками сохранен.")

Результаты классификации для сообщений 'Прочее':
message_type
Инфраструктурное    9680
Прикладное          9521
ZOD                  600
Тестовое             244
EMAIL                 99
Name: count, dtype: int64

Примеры сообщений типа 'Инфраструктурное':


,id,text,text_len,processed_text,message_type
44576,ebb45dc3-24c3-4669-ac35-f406e142e64c,Сервис: Сервис входа в БанкОнлайн\nИС сервиса:...,286,сервис сервис вход банконлайн иса сервис описа...,Инфраструктурное
1503,8d1858e7-37a9-455a-8895-c7d0f47ff0ab,Авария: EM-11504\nИС: SIEM\nОписание: 3001\nНа...,121,авария иса описание назначать группа,Инфраструктурное
41768,8a48e54d-2b55-4ff9-b473-3d23df7aeec5,Авария: [EM-20166] (https://servicedesk.ru/EM-...,244,авария иса описание знать сервис назначать гру...,Инфраструктурное
1073,cade7137-ee02-449c-940a-b50c9a133f01,Авария: EM-11074\nИС: GitLab CI\nОписание: 260...,118,авария иса описание назначать группа,Инфраструктурное
43753,f5e49c85-092d-4822-9f56-bd9a076662b1,Авария: [EM-50151] (https://servicedesk.ru/EM-...,255,авария иса описание учебный авария знать серви...,Инфраструктурное



Примеры сообщений типа 'Прикладное':


,id,text,text_len,processed_text,message_type
4359,ce7bb916-bb30-4e07-9cc5-4079691d0b6e,Авария: EM-14360\nИС: VPN Gateway\nОписание: T...,171,авария иса описание обращение соединение разры...,Прикладное
15783,ed957dd3-c400-4b2a-a017-7fcc8dc8c31f,Контур: ИФТ\nEM-11184 \nИС: JHA Core Banking \...,290,контур ифт иса описание проблема файловый сист...,Прикладное
26298,7169dad1-70b7-4796-a249-f4bbc2b2db15,Авария на сетевой инфраструктуре [EM-50199] (h...,138,авария сетевой инфраструктура пермь ул попов о...,Прикладное
22604,ec391e98-e523-475e-a38b-01fca2684292,Авария: EM-20505 \nИС: SMA Scheduler \nОписани...,394,авария иса описание недоступность авторизация ...,Прикладное
34617,1e9ccab6-143e-4c6e-abea-391c92b2068c,Сервис: Сервис формирования выписок\nИС сервис...,294,сервис сервис формирование выписка иса сервис ...,Прикладное



Примеры сообщений типа 'ZOD':


...


Примеры сообщений типа 'EMAIL':


,id,text,text_len,processed_text,message_type
40265,73f4ed0c-c177-4997-ba56-39e0a4f52580,[Отправить письмо](mailto:grigoriev.q@corp.bank),48,отправлять письмо,EMAIL
40279,7d2b9719-2970-4147-a0f2-bf647dfcf53e,[Отправить письмо](mailto:volkov.q@finance.org),47,отправлять письмо,EMAIL
40256,2fb3e95c-e719-42e9-a6f6-387e20139e7d,[Отправить письмо](mailto:petrov.c@intra.bank),46,отправлять письмо,EMAIL
40288,daa241a5-02c2-4163-b720-86521bc8338d,[Отправить письмо](mailto:zaharov.t@finance.org),48,отправлять письмо,EMAIL
40286,0399684c-a642-4e07-8f3e-e59c23f109e3,[Отправить письмо](mailto:pavlov.c@banking.local),49,отправлять письмо,EMAIL



Примеры сообщений типа 'Тестовое':


,id,text,text_len,processed_text,message_type
45140,4b455c11-b9e4-4185-ac45-cd874bb4bc54,тес товеое о повещение,22,тес товеое повещение,Тестовое
45237,7688e160-5059-42aa-8c11-e0d6b7eed94c,manuaйl_tt Опис:tset,20,й опис,Тестовое
45107,f246217d-fde1-46bf-b33c-04564e26bfc5,тестовое оповещени AVAR:е,25,тестовый оповещенить е,Тестовое
45322,131d0a29-67ff-4af8-bde9-06cbba6cfd79,тес товое оповещение,20,тес товый оповещение,Тестовое
45318,34cdf96b-4bde-4b73-b277-4eef88b84189,тет Опис:,9,тета опис,Тестовое



Датасет 'Прочее' с метками сохранен.


In [32]:
neww = other_df.text[other_df['message_type'] == 'Тестовое'] + other_df.processed_text[other_df['message_type'] == 'Тестовое']
neww.to_csv('../data/unknown.csv', index=False)

In [33]:
import pandas as pd

try:
    main_patterns_df = pd.read_csv('../data/final_labeled_dataset.csv')
    main_patterns_df.rename(columns={'pattern_name': 'label'}, inplace=True)
    print(f"Загружен датасет с основными паттернами: {len(main_patterns_df)} строк.")

    other_types_df = pd.read_csv('../data/other_classified_dataset.csv')
    other_types_df.rename(columns={'message_type': 'label'}, inplace=True)
    print(f"Загружен датасет с доп. классами: {len(other_types_df)} строк.")
    
except FileNotFoundError as e:
    print(f"Ошибка загрузки файла: {e}. Убедитесь, что все предыдущие шаги выполнены.")

final_columns = ['text', 'processed_text', 'label']

main_patterns_df = main_patterns_df[final_columns]
other_types_df = other_types_df[final_columns]

# pd.concat просто "склеивает" два датафрейма один под другим
training_dataset = pd.concat([main_patterns_df, other_types_df], ignore_index=True)

print(f"\nОбъединенный датасет создан. Общее количество строк: {len(training_dataset)}")

print("\nИтоговое распределение классов в тренировочном датасете:")
print(training_dataset['label'].value_counts())

print("\nСлучайные примеры из финального тренировочного датасета:")
display(training_dataset.sample(10))

# готовый к обучению датасет
training_dataset.to_csv('../data/TRAINING_DATASET.csv', index=False)
print("\nФИНАЛЬНЫЙ ТРЕНИРОВОЧНЫЙ ДАТАСЕТ сохранен в файл 'TRAINING_DATASET.csv'")

Загружен датасет с основными паттернами: 25698 строк.
Загружен датасет с доп. классами: 20144 строк.

Объединенный датасет создан. Общее количество строк: 45842

Итоговое распределение классов в тренировочном датасете:
label
Инфраструктурное                          9680
Прикладное                                9521
Высокая загрузка CPU                      3501
Сетевая недоступность (TLS/HTTPS/Ping)    2501
Проблемы с ФС (/pg_walarchive)            2324
Падение компонентов (Pods/Agents)         2230
API Авторизации (HTTP 500)                2124
Сбой сервиса PostgreSQL                   2014
Превышение порогов (Мониторинг)           1918
Сбой соединения с Oracle (ORA-12514)      1856
Отсутствие Heartbeat                      1696
Ошибка репликации PostgreSQL              1527
Широкомасштабный сбой                     1444
Проблемы с памятью (RAM/SWAP)              712
Проблемы с Zabbix-агентом                  672
ZOD                                        600
Проблемы с очередями / 

,text,processed_text,label
8765,Авария: EM-11658 \nИС: GitLab CI \nОписание: О...,авария иса описание ошибка репликация сечь хос...,Ошибка репликации PostgreSQL
12854,Авария: [EM-70321] (https://servicedesk.ru/EM-...,авария иса описание учебный авария отсутствова...,Отсутствие Heartbeat
16001,Авария 1 категории\n[EM-81468] (https://servic...,авария категория иса описание ошибка авторизац...,Высокая загрузка CPU
5542,Авария: EM-04064\nИС: SMA Scheduler\nОписание:...,авария иса описание порог количество сессия пр...,Превышение порогов (Мониторинг)
25317,Подозрение на широкомасштабный сбой по КЕ: Bac...,подозрение широкомасштабный сбой ке авария мин...,Широкомасштабный сбой
43895,Авария на сетевой инфраструктуре [EM-30431] (h...,авария сетевой инфраструктура санкт птербург н...,Инфраструктурное
35222,Контур: ИФТ\nEM-12033 \nИС: Payments API \nОпи...,контур ифт иса описание недоступный порт ошибк...,Инфраструктурное
34889,Контур: ИФТ\nEM-11700 \nИС: SIEM \nОписание: К...,контур ифт иса описание критический заполнение...,Инфраструктурное
38894,Авария: EM-31631 \nИС: Anti-Fraud Engine \nОпи...,авария иса описание высокий нагрузка замечать ...,Прикладное
17020,Авария 1 категории\n[EM-82487] (https://servic...,авария категория иса описание ошибка авторизац...,Высокая загрузка CPU



ФИНАЛЬНЫЙ ТРЕНИРОВОЧНЫЙ ДАТАСЕТ сохранен в файл 'TRAINING_DATASET.csv'


In [34]:
import pandas as pd

try:
    final_df = pd.read_csv('../data/TRAINING_DATASET.csv')
    print(f"Загружен полный датасет: {final_df.shape[0]} строк.")
except FileNotFoundError:
    print("Ошибка: Файл TRAINING_DATASET.csv не найден. Убедитесь, что предыдущий шаг выполнен.")
    exit()

# --- Создаем датасет для определенных паттернов ---
specific_labels_to_keep = ~final_df['label'].isin(['Прикладное', 'Инфраструктурное']) # Исключаем общие классы
df_model_1 = final_df[specific_labels_to_keep].copy()

# Сохраняем его
df_model_1.to_csv('../data/TRAINING_DATASET_SPECIFIC.csv', index=False)
print(f"\nДатасет для модели, которая будет обучаться на конкретных паттернах создан: {df_model_1.shape[0]} строк.")
print("Распределение классов в нем:")
print(df_model_1['label'].value_counts())


# --- Создаем датасет для общих классов ("Бинарный сортировщик") ---
general_labels_to_keep = final_df['label'].isin(['Прикладное', 'Инфраструктурное'])
df_model_2 = final_df[general_labels_to_keep].copy()

# Сохраняем его
df_model_2.to_csv('../data/TRAINING_DATASET_GENERAL.csv', index=False)
print(f"\nДатасет для Модели №2 (общие классы) создан: {df_model_2.shape[0]} строк.")
print("Распределение классов в нем:")
print(df_model_2['label'].value_counts())

Загружен полный датасет: 45842 строк.

Датасет для модели, которая будет обучаться на конкретных паттернах создан: 26641 строк.
Распределение классов в нем:
label
Высокая загрузка CPU                      3501
Сетевая недоступность (TLS/HTTPS/Ping)    2501
Проблемы с ФС (/pg_walarchive)            2324
Падение компонентов (Pods/Agents)         2230
API Авторизации (HTTP 500)                2124
Сбой сервиса PostgreSQL                   2014
Превышение порогов (Мониторинг)           1918
Сбой соединения с Oracle (ORA-12514)      1856
Отсутствие Heartbeat                      1696
Ошибка репликации PostgreSQL              1527
Широкомасштабный сбой                     1444
Проблемы с памятью (RAM/SWAP)              712
Проблемы с Zabbix-агентом                  672
ZOD                                        600
Проблемы с очередями / репликацией         577
Проблемы с местом на диске                 360
Тестовое                                   244
Ошибка записи на SMB-шару             